# 원인 분석 — Dimension #1: 업종 구성

[기획안_설계.md](기획안_설계.md) 6번 화면2 Section 1의 Dimension #1을 만드는 노트북.

**질문**: 위험 상권(299개, [02_EDA_상권쇠퇴분석.ipynb](02_EDA_상권쇠퇴분석.ipynb) STEP 2에서 도출)에 어떤 서비스업종이 유독 많이 몰려있는가?

**비교 방법**: 각 업종이 '위험 상권 안에서 차지하는 점포 비중'과 '전체 1,650개 상권에서 차지하는 점포 비중'을 비교해서, 위험 상권에 유난히 많은 업종을 찾는다.

**진행 방식**: 단계별로 나눠서 하나씩 진행.

---
## STEP 1. 위험 상권 리스트 재현 + 업종별 원본 데이터 로드

이 노트북은 새 파일이라 이전 노트북의 변수를 그대로 이어받을 수 없음. 위험 상권 판정에 필요한 최소한의 과정만 다시 계산한다.

재현해야 할 것 ([02_EDA_상권쇠퇴분석.ipynb](02_EDA_상권쇠퇴분석.ipynb) STEP 1·2 요약):
1. `stores` 원본 로드 + 컬럼 보정 (프랜차이즈_점포_수, 전체_점포_수, 개업_점포_수)
2. 상권×분기 집계 → KPI-1(순증감율), KPI-2(폐업률 증감) 계산
3. 최신 분기(20254) 기준 위험 상권(순증감율<0 AND 폐업률증감>0) 판정

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

DATA_DIR = "../data"
df_stores = pd.read_csv(f"{DATA_DIR}/seoul_commercial_district_stores_2025.csv", encoding="cp949")
df_master = pd.read_csv(f"{DATA_DIR}/seoul_commercial_district_master_2025.csv", encoding="cp949")

print("stores:", df_stores.shape)
print("master:", df_master.shape)

stores: (304775, 12)
master: (1650, 11)


In [2]:
# --- 컬럼 보정 (02번 노트북 STEP 1과 동일 로직) ---
df_stores_fixed = df_stores.copy()

# 1) 프랜차이즈_점포_수 보정: round(일반 * 프차 / 100)
전체_보정후_예상값 = (df_stores_fixed["일반_점포_수"] * df_stores_fixed["프랜차이즈_점포_수"] / 100).round(0).astype(int)
원래_공식_성립 = df_stores_fixed["전체_점포_수"] == (df_stores_fixed["일반_점포_수"] + df_stores_fixed["프랜차이즈_점포_수"])
df_stores_fixed["프랜차이즈_점포_수_보정"] = np.where(원래_공식_성립, df_stores_fixed["프랜차이즈_점포_수"], 전체_보정후_예상값)

# 2) 일반_점포_수 보정: 전체 - 프랜차이즈(보정), 음수는 0으로 클리핑
df_stores_fixed["일반_점포_수_보정"] = (df_stores_fixed["전체_점포_수"] - df_stores_fixed["프랜차이즈_점포_수_보정"]).clip(lower=0)

# 3) 전체_점포_수 보정: max(원본, 일반보정+프차보정)
전체_후보 = df_stores_fixed["일반_점포_수_보정"] + df_stores_fixed["프랜차이즈_점포_수_보정"]
df_stores_fixed["전체_점포_수_보정"] = np.maximum(df_stores_fixed["전체_점포_수"], 전체_후보)

# 4) 개업_점포_수 보정: 전체_점포_수_보정을 넘는 값만 클리핑
df_stores_fixed["개업_점포_수_보정"] = np.minimum(df_stores_fixed["개업_점포_수"], df_stores_fixed["전체_점포_수_보정"])

# 검증: 공식이 100% 맞는지 재확인 (02번 노트북에서 이미 확인했지만, 이 노트북에서도 재현이 맞는지 확인)
mismatch = df_stores_fixed[df_stores_fixed["전체_점포_수_보정"] != (df_stores_fixed["일반_점포_수_보정"] + df_stores_fixed["프랜차이즈_점포_수_보정"])]
print(f"보정 후 공식이 안 맞는 행: {len(mismatch)} / {len(df_stores_fixed)} (0이어야 함, 02번 노트북과 같은 결과가 나와야 재현이 맞는 것)")

보정 후 공식이 안 맞는 행: 0 / 304775 (0이어야 함, 02번 노트북과 같은 결과가 나와야 재현이 맞는 것)


In [3]:
# --- 상권×분기 집계 (업종은 아직 합치지 않음 — 이번 노트북에서는 업종별로 봐야 하므로 별도 보관) ---
group_keys = ["상권_코드", "상권_코드_명", "상권_구분_코드_명", "기준_년분기_코드"]
sum_cols_fixed = ["전체_점포_수_보정", "일반_점포_수_보정", "프랜차이즈_점포_수_보정", "개업_점포_수_보정", "폐업_점포_수"]

df_agg = df_stores_fixed.groupby(group_keys, as_index=False)[sum_cols_fixed].sum()
df_agg = df_agg.rename(columns={
    "전체_점포_수_보정": "전체_점포_수",
    "일반_점포_수_보정": "일반_점포_수",
    "프랜차이즈_점포_수_보정": "프랜차이즈_점포_수",
    "개업_점포_수_보정": "개업_점포_수",
})

print("상권×분기 집계:", df_agg.shape, "(02번 노트북과 같은 (6600, 9)여야 함)")

상권×분기 집계: (6600, 9) (02번 노트북과 같은 (6600, 9)여야 함)


In [4]:
# --- KPI-1(순증감율), KPI-2(폐업률 증감) 계산 후 위험 상권 판정 ---
분모_안전 = df_agg["전체_점포_수"].replace(0, np.nan)
df_agg["KPI1_순증감율"] = (df_agg["개업_점포_수"] - df_agg["폐업_점포_수"]) / 분모_안전
df_agg["폐업률"] = df_agg["폐업_점포_수"] / 분모_안전

df_agg = df_agg.sort_values(["상권_코드", "기준_년분기_코드"]).reset_index(drop=True)
df_agg["폐업률_전분기"] = df_agg.groupby("상권_코드")["폐업률"].shift(1)
df_agg["KPI2_폐업률증감"] = df_agg["폐업률"] - df_agg["폐업률_전분기"]

# 최신 분기(20254) 기준 위험 판정: 순증감율<0 AND 폐업률증감>0
df_최신분기 = df_agg[df_agg["기준_년분기_코드"] == 20254].copy()
위험_조건 = (df_최신분기["KPI1_순증감율"] < 0) & (df_최신분기["KPI2_폐업률증감"] > 0)
df_위험상권 = df_최신분기[위험_조건]

print(f"위험 상권 수: {len(df_위험상권)} (02번 노트북과 같은 299개여야 함, 전체 {len(df_최신분기)}개 중)")

위험 상권 수: 299 (02번 노트북과 같은 299개여야 함, 전체 1650개 중)


---
## STEP 2. 최신 분기 업종별 점포 수를 두 그룹으로 나눠 집계

업종 구성을 비교하려면 STEP 1에서는 뭉개졌던 `서비스_업종_코드_명`이 다시 필요함. 여기서는 `df_stores_fixed`(업종 단위, 보정 완료)에서 **최신 분기(20254)만** 골라서, 각 상권이 "위험 상권"인지 아닌지 표시(태그)한 뒤 업종별로 집계한다.

하는 일:
1. 최신 분기(20254) 업종 단위 데이터만 추출
2. 각 행(상권×업종)에 "이 상권이 위험 상권인가"를 표시하는 열 추가
3. 업종별 + 위험여부별로 점포 수 합산

In [5]:
# 최신 분기(20254)의 업종 단위(보정 완료) 데이터만 추출
df_최신분기_업종 = df_stores_fixed[df_stores_fixed["기준_년분기_코드"] == 20254].copy()

# 위험 상권 여부를 표시: 상권_코드가 df_위험상권에 있으면 True
위험_상권코드_집합 = set(df_위험상권["상권_코드"])
df_최신분기_업종["위험상권_여부"] = df_최신분기_업종["상권_코드"].isin(위험_상권코드_집합)

print("최신 분기 업종 단위 행 수:", len(df_최신분기_업종))
print("이 중 위험 상권에 속한 행:", df_최신분기_업종["위험상권_여부"].sum())
print()

# 검증: 위험상권_여부가 True인 행들의 고유 상권_코드 개수가 299개(위험 상권 수)와 같아야 함
print("위험상권_여부=True인 행의 고유 상권 수:", df_최신분기_업종.loc[df_최신분기_업종["위험상권_여부"], "상권_코드"].nunique(), "(299여야 함)")

최신 분기 업종 단위 행 수: 75985
이 중 위험 상권에 속한 행: 14457

위험상권_여부=True인 행의 고유 상권 수: 299 (299여야 함)


### 업종별 + 위험여부별 점포 수 집계

`groupby(["서비스_업종_코드_명", "위험상권_여부"])`로 두 기준을 동시에 묶어서, "이 업종이 위험 상권 그룹/전체(비위험) 그룹에 각각 점포가 몇 개 있는지"를 만든다.

In [6]:
# 업종별 x 위험여부별로 전체_점포_수_보정을 합산
# .unstack() : groupby 결과에서 두 번째 기준(위험상권_여부)을 행이 아니라 열로 펼쳐서
#              "업종 하나당 한 행, True/False 두 열"의 보기 편한 표로 바꿔줌
업종별_위험여부별_점포수 = (
    df_최신분기_업종
    .groupby(["서비스_업종_코드_명", "위험상권_여부"])["전체_점포_수_보정"]
    .sum()
    .unstack(fill_value=0)  # 특정 업종이 한쪽 그룹에 아예 없으면 결측 대신 0으로 채움
)

# 컬럼 이름을 알아보기 쉽게 변경
업종별_위험여부별_점포수.columns = ["비위험상권_점포수", "위험상권_점포수"]

print("업종 개수:", len(업종별_위험여부별_점포수))
print()
print("검증: 위험상권_점포수 합계가 위험 상권들의 전체_점포_수 합계와 같아야 함")
print("  업종별 집계 합:", 업종별_위험여부별_점포수["위험상권_점포수"].sum())
print("  위험상권 전체_점포_수 직접 합:", df_위험상권["전체_점포_수"].sum())

업종별_위험여부별_점포수.head(10)

업종 개수: 100

검증: 위험상권_점포수 합계가 위험 상권들의 전체_점포_수 합계와 같아야 함
  업종별 집계 합: 95448
  위험상권 전체_점포_수 직접 합: 95448


,비위험상권_점포수,위험상권_점포수
서비스_업종_코드_명,,
DVD방,354,92
PC방,749,220
가구,2250,492
가방,2486,540
가전제품,1682,370
가전제품수리,1599,361
가정용품임대,604,147
건축물청소,2908,698
게스트하우스,953,331


---
## STEP 3. 비중(비율)으로 정규화해서 비교

STEP 2의 원시 점포 수는 그대로 비교하면 안 됨 — 위험상권 그룹(299개 상권)과 비위험상권 그룹(1,351개 상권)은 애초에 크기가 다르므로, 점포 수가 많은 업종은 두 그룹 모두에서 절대값이 크게 나오는 게 당연함.

그래서 각 그룹 안에서 "이 업종이 전체 점포 수 중 몇 %를 차지하는가"로 바꿔야 공정한 비교가 됨:
- 위험상권_비중 = 위험상권_점포수 / 위험상권 전체 점포수 합계
- 비위험상권_비중 = 비위험상권_점포수 / 비위험상권 전체 점포수 합계

그 다음 두 비중의 차이(또는 상대비율)를 보면 "위험상권에 유독 많은/적은 업종"을 찾을 수 있음.

In [7]:
# 각 그룹의 전체 점포 수 합계 (분모)
위험상권_전체합 = 업종별_위험여부별_점포수["위험상권_점포수"].sum()
비위험상권_전체합 = 업종별_위험여부별_점포수["비위험상권_점포수"].sum()

print(f"위험상권 그룹 전체 점포 수: {위험상권_전체합}")
print(f"비위험상권 그룹 전체 점포 수: {비위험상권_전체합}")

# 업종별 비중 계산
df_업종비교 = 업종별_위험여부별_점포수.copy()
df_업종비교["위험상권_비중"] = df_업종비교["위험상권_점포수"] / 위험상권_전체합
df_업종비교["비위험상권_비중"] = df_업종비교["비위험상권_점포수"] / 비위험상권_전체합

# 검증: 비중 합계가 각각 1(100%)이 되어야 함
print()
print("위험상권_비중 합계 (1이어야 함):", df_업종비교["위험상권_비중"].sum())
print("비위험상권_비중 합계 (1이어야 함):", df_업종비교["비위험상권_비중"].sum())

위험상권 그룹 전체 점포 수: 95448
비위험상권 그룹 전체 점포 수: 378515

위험상권_비중 합계 (1이어야 함): 1.0
비위험상권_비중 합계 (1이어야 함): 1.0


### 비중 차이(격차) 계산 및 정렬

`위험상권_비중 - 비위험상권_비중`이 양수면 "위험상권에 더 많이 몰려있는 업종", 음수면 "위험상권에 상대적으로 적은 업종".

In [8]:
df_업종비교["비중_차이"] = df_업종비교["위험상권_비중"] - df_업종비교["비위험상권_비중"]

print("=== 위험상권에 상대적으로 많이 몰려있는 업종 Top 10 ===")
print((df_업종비교.sort_values("비중_차이", ascending=False).head(10)[
    ["위험상권_비중", "비위험상권_비중", "비중_차이"]
] * 100).round(2).astype(str) + "%")

print()
print("=== 위험상권에 상대적으로 적은 업종 Top 10 ===")
print((df_업종비교.sort_values("비중_차이", ascending=True).head(10)[
    ["위험상권_비중", "비위험상권_비중", "비중_차이"]
] * 100).round(2).astype(str) + "%")

=== 위험상권에 상대적으로 많이 몰려있는 업종 Top 10 ===
            위험상권_비중 비위험상권_비중  비중_차이
서비스_업종_코드_명                        
청과상            2.7%    1.28%  1.42%
육류판매          1.93%     1.0%  0.93%
일반의원          2.65%    1.88%  0.77%
세무사사무소        1.91%    1.37%  0.54%
변호사사무소        1.36%    0.84%  0.52%
수산물판매         1.23%     0.9%  0.33%
부동산중개업        5.72%    5.43%  0.29%
자동차부품         0.76%    0.51%  0.26%
신발            1.16%    0.93%  0.23%
변리사사무소        0.34%    0.15%  0.19%

=== 위험상권에 상대적으로 적은 업종 Top 10 ===
            위험상권_비중 비위험상권_비중   비중_차이
서비스_업종_코드_명                         
일반의류          9.56%   12.13%  -2.57%
조명용품           1.6%    2.18%  -0.57%
시계및귀금속        0.73%    1.31%  -0.57%
컴퓨터및주변장치판매    1.23%    1.66%  -0.43%
일반교습학원        2.59%    3.01%  -0.42%
미용실           2.92%    3.18%  -0.26%
인테리어          0.56%    0.78%  -0.22%
미곡판매           0.3%    0.46%  -0.16%
커피-음료         3.42%    3.58%  -0.16%
예술품           0.29%    0.44%  -0.14%


---
## STEP 4. 상대비율로 재확인 + 표본 필터링

STEP 3의 %p 차이는 원래 비중이 큰 업종(예: 일반의류 9~12%)에서 절대값이 커 보이기 쉬움. "몇 배 더/덜 나타나는가"(상대비율)로 보면 원래 비중 크기와 무관하게 공정히 비교 가능.

또한 위험상권 그룹은 점포 수가 95,448개뿐이라, 원래 점포 수가 적은 업종(예: 위험상권 안에 5개뿐인 업종)은 우연히 비중이 튀어보일 수 있음 → **최소 표본 기준**을 두고 그 이하는 제외.

In [9]:
# 최소 표본 기준을 정하기 전에, 위험상권_점포수의 실제 분포를 먼저 확인
print("위험상권_점포수 분포:")
print(df_업종비교["위험상권_점포수"].describe())
print()
print("위험상권_점포수가 30 미만인 업종 개수:", (df_업종비교["위험상권_점포수"] < 30).sum(), "/ 100")
print("위험상권_점포수가 50 미만인 업종 개수:", (df_업종비교["위험상권_점포수"] < 50).sum(), "/ 100")

위험상권_점포수 분포:
count     100.000000
mean      954.480000
std      1449.225495
min        20.000000
25%       203.500000
50%       551.500000
75%      1150.750000
max      9128.000000
Name: 위험상권_점포수, dtype: float64

위험상권_점포수가 30 미만인 업종 개수: 4 / 100
위험상권_점포수가 50 미만인 업종 개수: 5 / 100


### 최소 표본 기준 적용 + 상대비율 계산

위험상권_점포수 최솟값이 20이고 50 미만인 업종이 5개뿐이라 극단적으로 작은 표본은 없지만, 가벼운 안전장치로 **위험상권_점포수 ≥ 50**인 업종만 남기고 상대비율을 계산한다.

상대비율 = 위험상권_비중 / 비위험상권_비중. 1보다 크면 위험상권에 더 많이, 1보다 작으면 더 적게 나타난다는 뜻.

In [10]:
# 최소 표본 기준 적용
최소_표본 = 50
df_업종비교_필터 = df_업종비교[df_업종비교["위험상권_점포수"] >= 최소_표본].copy()
print(f"필터 적용 후 남은 업종 수: {len(df_업종비교_필터)} / 100 (제외된 업종: {100 - len(df_업종비교_필터)}개)")

# 상대비율 계산: 위험상권_비중이 비위험상권_비중의 몇 배인지
df_업종비교_필터["상대비율"] = df_업종비교_필터["위험상권_비중"] / df_업종비교_필터["비위험상권_비중"]

print()
print("=== 상대비율 기준, 위험상권에 유독 많은 업종 Top 10 ===")
print(df_업종비교_필터.sort_values("상대비율", ascending=False).head(10)[
    ["위험상권_점포수", "위험상권_비중", "비위험상권_비중", "상대비율"]
])

필터 적용 후 남은 업종 수: 95 / 100 (제외된 업종: 5개)

=== 상대비율 기준, 위험상권에 유독 많은 업종 Top 10 ===
             위험상권_점포수   위험상권_비중  비위험상권_비중      상대비율
서비스_업종_코드_명                                        
변리사사무소            322  0.003374  0.001495  2.256086
청과상              2577  0.026999  0.012792  2.110600
육류판매             1842  0.019298  0.009994  1.930943
변호사사무소           1295  0.013568  0.008396  1.615966
자동차부품             729  0.007638  0.005054  1.511224
일반의원             2530  0.026507  0.018800  1.409941
법무사사무소            568  0.005951  0.004238  1.404301
세무사사무소           1824  0.019110  0.013677  1.397214
게스트하우스            331  0.003468  0.002518  1.377372
수산물판매            1177  0.012331  0.009035  1.364792


In [11]:
print("=== 상대비율 기준, 위험상권에 유독 적은 업종 Top 10 ===")
print(df_업종비교_필터.sort_values("상대비율", ascending=True).head(10)[
    ["위험상권_점포수", "위험상권_비중", "비위험상권_비중", "상대비율"]
])

=== 상대비율 기준, 위험상권에 유독 적은 업종 Top 10 ===
              위험상권_점포수   위험상권_비중  비위험상권_비중      상대비율
서비스_업종_코드_명                                         
중고차판매               60  0.000629  0.001519  0.413809
악기                  67  0.000702  0.001556  0.451103
시계및귀금속             701  0.007344  0.013056  0.562512
미곡판매               282  0.002954  0.004560  0.647925
재생용품 판매점           100  0.001048  0.001583  0.662048
예술품                279  0.002923  0.004367  0.669341
인테리어               537  0.005626  0.007846  0.717025
조명용품              1531  0.016040  0.021753  0.737362
컴퓨터및주변장치판매        1178  0.012342  0.016599  0.743523
자전거 및 기타운송장비        93  0.000974  0.001273  0.765160
